# Modul 2 — CLUSTER (Mahasiswa): Lab 1–4 Spark Architecture
Kelompok 5 mahasiswa: **1 PC Master + 4 PC Worker**. Notebook ini dijalankan oleh **setiap anggota** (PC Anda = Driver).
Pastikan Master kelompok sudah menjalankan `Modul2-Master.ipynb` dan 4 Worker sudah join (cek `http://<IP-Master>:8080`).

**Yang perlu diubah hanya `MASTER_IP` di baris pertama Sel 1.**
Latihan sendiri di 1 laptop? Start Master + 1 Worker di laptop itu, lalu isi `MASTER_IP` dengan IP laptop. Semua core otomatis dipakai.

Jalankan sel **berurutan dari atas**. Sel 1 cukup sekali; Lab boleh diulang.

| Sel | Isi | Konsep |
|---|---|---|
| Sel 1 | Koneksi ke cluster (membuat **Driver**) + fungsi bantu | Driver, Master, Executor |
| Lab 1 | Anatomi cluster: 16 task tersebar ke worker | Worker, Executor, Task |
| Lab 2 | Partisi vs core: prediksi waktu | Parallelism = min(partisi, core) |
| Lab 3 | Lazy evaluation, DAG, Job, Stage, Shuffle | Job, Stage, DAG |
| Lab 4 | Laporan penjualan e-commerce (DataFrame, 3 juta baris) | Exchange = shuffle |
| Penutup | `spark.stop()` → core dikembalikan ke cluster | |

**Melihat DAG:** buka Spark UI Driver `http://<IP-PC-Anda>:4040` (link tercetak di Sel 1) → tab **Jobs** → klik job (label `LAB1`, `LAB3`, …) → **DAG Visualization**.
DAG *tidak* ada di Master UI (8080). Setelah `spark.stop()` gunakan History Server (petunjuk di akhir).

Jika muncul error *Cannot call methods on a stopped SparkContext*: **Kernel → Restart Kernel**, lalu jalankan ulang dari Sel 1.

In [1]:
# ===== SEL 1: Koneksi ke cluster (membuat DRIVER) + fungsi bantu =====
MASTER_IP = "26.66.202.158"   # <-- SATU-SATUNYA YANG PERLU DIUBAH: IP PC Master kelompok
# ------------------------------------------------------------------------------

import os, sys, socket, time, json, uuid, math, random, urllib.request
from collections import Counter
from pathlib import Path

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MASTER = f"spark://{MASTER_IP}:7077"
REPORT_URL = f"http://{MASTER_IP}:8090"   # server laporan (Modul2-Master.ipynb)
ANGGOTA = 5            # 1 kelompok = 5 mahasiswa (dipakai untuk membagi core)
EXECUTOR_CORES = 1     # 1 core per executor -> kerja menyebar ke banyak worker
EXECUTOR_MEMORY = "1g"

# ---- Baca kapasitas cluster dari Master UI (:8080) ----
try:
    with urllib.request.urlopen(f"http://{MASTER_IP}:8080/json/", timeout=3) as r:
        _alive = [w for w in json.load(r).get("workers", []) if w.get("state") == "ALIVE"]
except Exception:
    _alive = []
N_WORKER = len(_alive)
TOTAL_CORES = sum(w.get("cores", 0) for w in _alive)

# Pembagian core otomatis (modul Bab 5.3):
#   1 worker (latihan sendiri, Master+Worker di 1 PC) -> pakai semua core
#   banyak worker (kelompok)                          -> total core / jumlah anggota
if N_WORKER <= 1:
    MAX_CORES = max(1, TOTAL_CORES)
else:
    MAX_CORES = max(1, TOTAL_CORES // ANGGOTA)

# ---- IP PC ini (Driver), terdeteksi otomatis ----
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.connect((MASTER_IP, 7077))
IP = sock.getsockname()[0]
sock.close()
HOST = socket.gethostname()

# Folder event log: agar DAG tetap bisa dilihat setelah spark.stop() (History Server)
EVENT_DIR = Path("spark-events").resolve()
EVENT_DIR.mkdir(exist_ok=True)

# Bersihkan SparkContext lama yang sudah mati (penyebab error "stopped SparkContext")
old_sc = SparkContext._active_spark_context
if old_sc is not None:
    try:
        old_sc.stop()
    except Exception:
        pass
SparkSession._instantiatedSession = None
SparkSession._activeSession = None

import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

builder = (
    SparkSession.builder
    .appName(f"Modul2-{HOST}")
    .master(MASTER)
    .config("spark.driver.host", IP)
    .config("spark.driver.bindAddress", IP)
    .config("spark.cores.max", str(MAX_CORES))
    .config("spark.executor.cores", str(EXECUTOR_CORES))
    .config("spark.executor.memory", EXECUTOR_MEMORY)
    # Python Driver & Worker sama (hindari error beda versi Python)
    .config("spark.pyspark.python", "python")
    .config("spark.pyspark.driver.python", "python")
    # Spark UI (tempat DAG Visualization) di port 4040
    .config("spark.ui.enabled", "true")
    .config("spark.driver.port", "7001")
    .config("spark.blockManager.port", "7002")
    .config("spark.ui.port", "4040")
)

# Event log (History Server). Di Windows butuh HADOOP_HOME\bin\winutils.exe;
# tanpa itu Spark gagal start ("HADOOP_HOME and hadoop.home.dir are unset").
HADOOP_HOME = os.environ.get("HADOOP_HOME", "")
EVENT_LOG = os.name != "nt" or os.path.isfile(os.path.join(HADOOP_HOME, "bin", "winutils.exe"))
if EVENT_LOG:
    import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

builder = (builder
               .config("spark.eventLog.enabled", "true")
               .config("spark.eventLog.dir", EVENT_DIR.as_uri()))

spark = builder.getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

# Alamat Spark UI pakai IP (jika 4040 sudah dipakai, Spark pindah ke 4041, dst.)
UI = f"http://{IP}:{(sc.uiWebUrl or ':4040').rsplit(':', 1)[-1]}"


# ---------------- Fungsi bantu ----------------
def tandai(deskripsi):
    """Beri label Job berikutnya (kata pertama = group) agar mudah dicari di tab Jobs Spark UI."""
    sc.setJobGroup(deskripsi.split()[0], deskripsi)


def jumlah_executor():
    """Executor yang sudah terdaftar ke Driver (tidak termasuk Driver sendiri)."""
    return sc._jsc.sc().getExecutorMemoryStatus().size() - 1


def core_aplikasi():
    """Core yang BENAR-BENAR didapat aplikasi ini."""
    return jumlah_executor() * EXECUTOR_CORES


LAPOR = True   # dimatikan otomatis jika monitor tidak terjangkau (agar task tidak melambat)


def report(run, app, nomor, status):
    """Dijalankan di EXECUTOR: kirim status task ke monitor."""
    if not LAPOR:
        return False
    data = {
        "run": run,
        "app": app,
        "task": nomor,
        "driver": HOST,                     # milik mahasiswa siapa
        "worker": socket.gethostname(),     # dikerjakan di PC mana
        "pid": os.getpid(),
        "status": status,
    }
    req = urllib.request.Request(
        REPORT_URL,
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=2) as response:
            return response.status == 200
    except Exception as e:
        print(f"Gagal melapor ke Master: {e}")   # muncul di log Worker
        return False


def pantau(rdd, label, jeda=0):
    """Bungkus RDD: tiap partisi (= 1 task) melapor RUNNING saat mulai dan DONE saat selesai.
    Data dialirkan (tidak ditampung di memori), dan pantau() tetap LAZY seperti transformation lain."""
    run = str(uuid.uuid4())

    def proses_partition(idx, isi):
        report(run, label, idx, "RUNNING")
        if jeda:
            time.sleep(jeda)                 # agar RUNNING terlihat di monitor
        for x in isi:
            yield x
        report(run, label, idx, "DONE")

    return rdd.mapPartitionsWithIndex(proses_partition)


# ---- Daftarkan Driver ke monitor (agar panel [B]/[C] bisa membaca Spark UI ini) ----
try:
    req = urllib.request.Request(
        REPORT_URL + "/driver",
        data=json.dumps({"app_id": sc.applicationId, "host": HOST, "ip": IP, "ui": UI}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    urllib.request.urlopen(req, timeout=3).close()
    with urllib.request.urlopen(REPORT_URL, timeout=3) as r:
        print("Monitor       :", r.read().decode().strip())
except Exception as e:
    LAPOR = False
    print("Info: monitor tidak terjangkau, laporan task dimatikan ->", e)

# Executor butuh beberapa detik untuk terdaftar
_t0 = time.time()
while jumlah_executor() < MAX_CORES // EXECUTOR_CORES and time.time() - _t0 < 30:
    time.sleep(1)

print("=" * 64)
print("Saya DRIVER   :", HOST, f"({IP})")
print("Spark Master  :", sc.master)
print("Spark Version :", spark.version)
print("Application ID:", sc.applicationId)
print(f"Cluster       : {N_WORKER} worker ALIVE, {TOTAL_CORES} core total")
print(f"Aplikasi ini  : {jumlah_executor()} executor / {core_aplikasi()} core (maks {MAX_CORES})")
print("Spark UI (DAG):", UI, "-> tab Jobs / Stages -> 'DAG Visualization'")
print("Event log     :", EVENT_DIR if EVENT_LOG else "OFF (winutils.exe tidak ada; DAG tetap di port 4040)")
if N_WORKER == 0:
    print("\nPERINGATAN: Master tidak terjangkau / belum ada worker. Cek MASTER_IP, Master & Worker sudah start?")
elif jumlah_executor() == 0:
    print("\nPERINGATAN: belum dapat executor. Aplikasi WAITING (core habis) "
          "atau worker tidak bisa menghubungi PC ini (firewall).")

Monitor       : Monitor aktif. Driver terdaftar: 4 | Laporan task: 0
Saya DRIVER   : Leyzu (26.152.94.182)
Spark Master  : spark://26.66.202.158:7077
Spark Version : 3.5.3
Application ID: app-20261023234640-0003
Cluster       : 4 worker ALIVE, 36 core total
Aplikasi ini  : 7 executor / 7 core (maks 7)
Spark UI (DAG): http://26.152.94.182:4040 -> tab Jobs / Stages -> 'DAG Visualization'
Event log     : C:\Spark\notebooks\spark-events


## LAB 1 — Anatomi Cluster
**Tujuan:** melihat Driver, Master, Worker, Executor, dan Task bekerja; membuktikan task dikerjakan PC yang berbeda.

**Perkiraan hasil:** 16 task tersebar ke **paling banyak 4 worker** kelompok (bisa termasuk PC teman). Waktu ≈ `ceil(16 / core) × 4 detik`: 2 core → 32 s, 3 core → 24 s, 4 core → 16 s, 8 core → 8 s (+ overhead).

**Diskusi:** (1) Tunjuk PC mana Driver, Master, Worker di kelompok Anda. (2) Mengapa jumlah worker terpakai maksimal 4, bukan 16? (3) Mengapa total ≈ 16 detik padahal 1 task 4 detik? (4) Apa yang terjadi pada Executor setelah `spark.stop()`?

In [2]:
# ===== LAB 1: Anatomi cluster (Driver, Master, Worker, Executor, Task) =====
print("=" * 64)
print("LAB 1 - ANATOMI CLUSTER")
print("=" * 64)
print("Saya DRIVER             :", HOST, f"({IP})")
print("Cluster manager (Master):", sc.master)
print("Application ID          :", sc.applicationId)
print("Core yang saya dapat    :", core_aplikasi())

N_TASKS = 16
rdd = sc.parallelize(range(N_TASKS), N_TASKS).setName("angka_16")   # 16 partisi -> 16 task
print("Jumlah partisi          :", rdd.getNumPartitions(), f"-> {N_TASKS} task")


def do_work(nomor):
    time.sleep(4)   # pura-pura bekerja agar terlihat di monitor dosen
    return (nomor, socket.gethostname(), os.getpid())


print("\nMengirim pekerjaan ke cluster... lihat layar master!")
tandai("LAB1 Anatomi - 16 task")
t0 = time.time()
hasil = pantau(rdd.map(do_work), "Lab1-Anatomi").collect()   # collect() = ACTION -> job dimulai
elapsed = time.time() - t0

print(f"\n{'TASK':<8}{'DIKERJAKAN OLEH (WORKER)':<30}{'PID'}")
for nomor, worker, pid in sorted(hasil):
    print(f"{nomor:<8}{worker:<30}{pid}")

per_worker = Counter(w for _, w, _ in hasil)
print("\nRINGKASAN")
print("Jumlah worker terpakai :", len(per_worker))
for w, n in sorted(per_worker.items()):
    print(f"  - {w:<28}{n} task")
print(f"Total waktu            : {elapsed:.1f} detik")

LAB 1 - ANATOMI CLUSTER
Saya DRIVER             : Leyzu (26.152.94.182)
Cluster manager (Master): spark://26.66.202.158:7077
Application ID          : app-20261023234640-0003
Core yang saya dapat    : 7
Jumlah partisi          : 16 -> 16 task

Mengirim pekerjaan ke cluster... lihat layar master!


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 4 times, most recent failure: Lost task 0.3 in stage 0.0 (TID 14) (26.204.199.100 executor 4): java.io.IOException: Cannot run program "C:\Spark\config\..\env\Scripts\python.exe": CreateProcess error=2, The system cannot find the file specified
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1170)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1089)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:181)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scal

## LAB 2 — Partisi, Core, dan Parallelism
**Tujuan:** membuktikan **parallelism = min(partisi, core)** dan memprediksi waktu.
Tulis **tebakan waktu Anda dulu** sebelum melihat hasil.

Kolom perkiraan dihitung otomatis dari core yang Anda dapat. Contoh:

| Partisi | 1 task | Perkiraan 2 core | Perkiraan 4 core | Perkiraan 8 core |
|---|---|---|---|---|
| 1 | 32 s | ~32 s | ~32 s | ~32 s |
| 4 | 8 s | ~16 s | ~8 s | ~8 s |
| 8 | 4 s | ~16 s | ~8 s | ~4 s |
| 32 | 1 s | ~16 s | ~8 s | ~4 s |

**Lanjutan (tugas mandiri no. 3 modul):** bandingkan hasil saat 5 anggota jalan bersamaan dengan latihan sendiri (Master + 1 Worker di 1 laptop, semua core terpakai).

In [ ]:
# ===== LAB 2: Partisi, core, dan parallelism =====
NUM_ITEMS = 32
SECONDS_PER_ITEM = 1
CORES = max(1, core_aplikasi())   # core yang BENAR-BENAR didapat aplikasi ini


def work(x):
    time.sleep(SECONDS_PER_ITEM)   # tiap item butuh 1 detik
    return x


def run_test(n_partitions):
    tandai(f"LAB2 {n_partitions} partisi")
    t0 = time.time()
    rdd = sc.parallelize(range(NUM_ITEMS), n_partitions).map(work)
    pantau(rdd, f"Lab2-{n_partitions:>2} partisi").collect()
    return time.time() - t0


# Pemanasan: proses Python di executor dibuat dulu agar tidak mengotori pengukuran pertama
tandai("LAB2 pemanasan")
sc.parallelize(range(CORES * 2), CORES * 2).map(lambda x: x).count()

print("=" * 64)
print(f"LAB 2 - PARALLELISM (item = {NUM_ITEMS}, {SECONDS_PER_ITEM} s/item, core aplikasi ini = {CORES})")
print("=" * 64)
print("Tebak waktunya dulu, sebelum melihat hasil!\n")
print(f"{'PARTISI':<10}{'TASK':<7}{'GELOMBANG':<11}{'PERKIRAAN(s)':<14}{'HASIL(s)'}")

for p in [1, 4, 8, 32]:
    per_task = (NUM_ITEMS / p) * SECONDS_PER_ITEM   # durasi 1 task
    waves = math.ceil(p / CORES)                    # berapa "putaran"
    estimate = waves * per_task
    actual = run_test(p)
    print(f"{p:<10}{p:<7}{waves:<11}{estimate:<14.1f}{actual:.1f}")

print("\nKESIMPULAN: kecepatan dibatasi oleh yang LEBIH KECIL antara jumlah partisi dan jumlah core.")
print("Hasil nyata biasanya 1-3 detik lebih lama karena overhead penjadwalan & jaringan.")

## LAB 3 — Lazy Evaluation, DAG, Job, Stage, Shuffle
Lab ini dipecah **3 sel** agar sifat *lazy* terlihat.

1. **Sel 3a** hanya menulis rencana (transformation). Cek Spark UI tab Jobs: **tidak ada job LAB3**. Dosen: panel [C]/[D] juga diam.
2. **Sel 3b** `collect()` → **1 job, 2 stage**: Stage 0 (8 task: buat data, flatMap, map, tulis shuffle) dan Stage 1 (reduceByKey, 8 task).
3. **Sel 3c** `count()` → job baru; Stage 0 **SKIPPED** karena output shuffle masih tersimpan, jadi jauh lebih cepat.

Hasil word count: 10 kata unik, masing-masing sekitar 1,44 juta (data acak).
Jika terlalu lambat, kecilkan `LINES_PER_PARTITION` menjadi `100_000`.

In [ ]:
# ===== LAB 3a: TRANSFORMATION saja (rencana / DAG, belum ada pekerjaan) =====
VOCAB = ["spark", "driver", "worker", "executor", "task",
         "stage", "partition", "cluster", "shuffle", "data"]
LINES_PER_PARTITION = 300_000


def make_lines(partition_number):
    """Data dibuat LANGSUNG di executor (tidak dikirim dari Driver)."""
    rnd = random.Random(partition_number)
    for _ in range(LINES_PER_PARTITION):
        yield " ".join(rnd.choice(VOCAB) for _ in range(6))


lines = pantau(sc.parallelize(range(8), 8).flatMap(make_lines),
               "Lab3-Stage0 buat data+flatMap+map").setName("baris")       # 8 partisi
words = lines.flatMap(lambda l: l.split(" ")).setName("kata")              # narrow
pairs = words.map(lambda w: (w, 1)).setName("pasangan")                    # narrow
counts = pantau(pairs.reduceByKey(lambda a, b: a + b),                     # WIDE -> shuffle
                "Lab3-Stage1 reduceByKey").setName("hitung_kata")

print("=" * 64)
print("LAB 3 - DAG, JOB, STAGE")
print("=" * 64)
print("Rencana (DAG) sudah ditulis, tapi BELUM ada pekerjaan di cluster (lazy).")
print("Job LAB3 sejauh ini:", len(sc.statusTracker().getJobIdsForGroup("LAB3")), "-> buktinya 0")
print(f"Cek {UI}/jobs/ : belum ada job berlabel LAB3.")

print("\nLINEAGE / DAG (perubahan indentasi '+-' = batas stage, ShuffledRDD = shuffle):")
print(counts.toDebugString().decode())

In [ ]:
# ===== LAB 3b: ACTION #1 -> JOB pertama =====
tandai("LAB3 collect() - wordcount")
t0 = time.time()
result = counts.collect()
print(f"Action #1 collect() selesai dalam {time.time() - t0:.1f} detik")
for k, v in sorted(result, key=lambda x: -x[1]):
    print(f"  {k:<10}{v:,}")

st = sc.statusTracker()
for jid in sorted(st.getJobIdsForGroup("LAB3")):
    info = st.getJobInfo(jid)
    print(f"\nJob {jid}: status {info.status}, stage {list(info.stageIds)}")

In [ ]:
# ===== LAB 3c: ACTION #2 -> JOB kedua (Stage 0 di-SKIP) =====
tandai("LAB3 count() - jumlah kata unik")
t0 = time.time()
n_unique = counts.count()
print(f"Action #2 count() selesai dalam {time.time() - t0:.1f} detik -> {n_unique} kata unik")

st = sc.statusTracker()
for jid in sorted(st.getJobIdsForGroup("LAB3")):
    info = st.getJobInfo(jid)
    print(f"Job {jid}: status {info.status}, stage {list(info.stageIds)}")

print(f"\nBuka {UI}/jobs/ dan /stages/ : berapa Job, Stage, Task? Stage mana yang 'skipped'?")

**Diskusi Lab 3:** (1) Mengapa tidak ada job tepat setelah baris `reduceByKey` ditulis? (2) Mana transformation narrow dan mana wide? (3) Kode minggu lalu (40 partisi, 1 `collect`) menghasilkan berapa job/stage/task? (4) Jika `reduceByKey` diganti `groupByKey().mapValues(sum)`, apa yang berubah pada Shuffle Read/Write?

> Catatan: di PySpark, beberapa operasi narrow berturut-turut (`flatMap` + `map` + `mapPartitionsWithIndex`) digabung menjadi **satu kotak PythonRDD** di DAG, jadi kotak bisa lebih sedikit daripada operasi. Batas stage (shuffle) tetap jelas.

## LAB 4 — Studi Kasus Industri: Laporan Penjualan E-commerce
3 juta transaksi dibuat **langsung di cluster** (tanpa file), lalu diagregasi per kategori dan kota dengan DataFrame.

**Perkiraan hasil:** 12 partisi data → stage pembuatan data 12 task. Di `explain()` cari kata **Exchange** (= shuffle dari `groupBy`/`orderBy`). Tabel 5 kategori dan 6 kota; angka sama tiap run karena `rand(42)`.
Satu perintah DataFrame bisa menghasilkan lebih dari satu job/stage karena optimizer (Catalyst & AQE); itu normal.

In [ ]:
# ===== LAB 4: Laporan penjualan e-commerce (DataFrame) =====
spark.conf.set("spark.sql.shuffle.partitions", "8")   # default 200 terlalu banyak untuk kelas

N = 3_000_000
CATEGORIES = ["Electronics", "Fashion", "Food", "Household", "Books"]
CITIES = ["Jakarta", "Bandung", "Surabaya", "Yogyakarta", "Medan", "Makassar"]

# Transaksi disimulasikan langsung di cluster (3 juta baris, 12 partisi)
transactions = (
    spark.range(0, N, numPartitions=12)
    .withColumn("category", F.element_at(
        F.array(*[F.lit(k) for k in CATEGORIES]),
        (F.pmod(F.hash(F.col("id")), F.lit(5)) + 1).cast("int")))
    .withColumn("city", F.element_at(
        F.array(*[F.lit(k) for k in CITIES]),
        (F.pmod(F.hash(F.col("id"), F.lit(7)), F.lit(6)) + 1).cast("int")))
    .withColumn("amount", (F.rand(42) * 490000 + 10000).cast("int"))
)

print("=" * 64)
print("LAB 4 - LAPORAN PENJUALAN E-COMMERCE (3 juta transaksi)")
print("=" * 64)
print("Jumlah partisi data:", transactions.rdd.getNumPartitions())

# Catatan: jangan beri nama variabel "report" karena akan menimpa fungsi report() untuk monitor
laporan_kategori = (
    transactions.groupBy("category")
    .agg(F.count("*").alias("num_transactions"),
         F.round(F.sum("amount") / 1_000_000, 1).alias("revenue_million_idr"),
         F.round(F.avg("amount")).alias("avg_idr"))
    .orderBy(F.desc("revenue_million_idr"))
)

print("\nEXECUTION PLAN (cari kata 'Exchange' = SHUFFLE = batas stage):")
laporan_kategori.explain()

tandai("LAB4 laporan per kategori")
print("\nLAPORAN PER KATEGORI:")
laporan_kategori.show()

tandai("LAB4 laporan per kota")
print("LAPORAN PER KOTA:")
(transactions.groupBy("city")
 .agg(F.count("*").alias("num_transactions"),
      F.round(F.sum("amount") / 1_000_000, 1).alias("revenue_million_idr"))
 .orderBy(F.desc("revenue_million_idr"))
 .show())

print(f"Cek Spark UI -> tab Jobs / Stages / SQL di {UI}")

In [ ]:
# ===== PENUTUP (1/2): selama sel ini belum dijalankan ke-2, aplikasi masih hidup =====
# Buka Spark UI:
#   Jobs   : cari label LAB1..LAB4, klik job -> 'DAG Visualization'
#   Stages : DAG per stage, Shuffle Read/Write, stage SKIPPED (Lab 3)
#   SQL    : rencana Lab 4 (Exchange)
#   Executors: executor di worker mana saja
print("Spark UI:", UI)
print("Jika sudah selesai mengamati, jalankan sel berikutnya untuk spark.stop().")

In [ ]:
# ===== PENUTUP (2/2): kembalikan core ke cluster =====
spark.stop()
print("Spark dihentikan. Executor dimatikan Master, core kembali bebas.")

## Melihat DAG setelah `spark.stop()` (History Server)
**Khusus Windows:** event log hanya aktif jika `winutils.exe` ada di `%HADOOP_HOME%\bin`
(unduh `winutils.exe` + `hadoop.dll` sesuai versi Hadoop Spark Anda, set environment variable `HADOOP_HOME`, lalu restart Jupyter).
Tanpa itu, DAG tetap bisa dilihat di port 4040 selama aplikasi belum di-`stop()`.

Event log tersimpan di folder `spark-events` (di samping notebook ini). Jalankan di CMD:

```bat
cd C:\spark\bin
spark-class -Dspark.history.fs.logDirectory=file:///C:/path/ke/folder/spark-events org.apache.spark.deploy.history.HistoryServer
```

Ganti path dengan yang tercetak pada baris **Event log** di Sel 1 (pakai `/`, bukan `\`).
Lalu buka `http://localhost:18080` → klik aplikasi → tab **Jobs** → **DAG Visualization**.

## Troubleshooting singkat (Bab 8 modul)
| Gejala | Solusi |
|---|---|
| `0 executor`, aplikasi WAITING | Core kelompok habis → minta anggota lain menjalankan sel `spark.stop()` |
| `0 worker ALIVE` | `MASTER_IP` salah, Master/Worker belum start, atau port 8080 diblokir |
| IP PC ini terdeteksi salah (VPN, WiFi + LAN) | Matikan jaringan lain, atau ganti `IP = sock.getsockname()[0]` dengan IP manual |
| Executor muncul lalu hilang / *Failed to connect to driver* | Cek `IP` di Sel 1 benar; buka firewall untuk Python/Java di PC ini |
| *Python in worker has different version* | Python semua PC harus sama (path `sys.executable` juga harus ada di worker) |
| Monitor dosen kolom EXEC `-` | Port 4040 PC ini diblokir firewall |
| Lab 2 jauh dari perkiraan | Lihat "core aplikasi ini"; ulangi saat cluster sepi |
| *Cannot call methods on a stopped SparkContext* | Kernel → Restart Kernel, jalankan dari Sel 1 |